# CRAG Qualitative Error Analysis & Taxonomy — Objective 5
*Categorize failure cases into a taxonomy* (proposal: retrieval failure, extraction failure, hallucination despite correct retrieval).

This notebook takes every **incorrectly answered** question from a results file and auto-classifies it, then lets you hand-review and export a labelled CSV for the writeup.

### Taxonomy (proposal categories + emergent ones)
| Category | Definition | Detection |
|---|---|---|
| **retrieval_failure** | answer absent from the docs the model saw | gold not in any retrieved doc |
| **extraction_failure** | answer was in the docs but model didn't extract it (said "I don't know") | gold in docs, but refusal |
| **hallucination** | docs had the answer, model gave a *different confident* answer | gold in docs, wrong non-refusal answer |
| **grader_misroute** | local docs had the answer, but grader said Incorrect -> sent to web -> lost | gold in local docs AND path went to websearch |
| **noisy_gold** | gold label itself is wrong/ambiguous (manual flag) | hand-labelled on review |

### Two modes
- **Mode A (GPU):** re-retrieves docs per question to check where the answer was -> full retrieval/extraction/hallucination split. Needs the pipeline loaded.
- **Mode B (CPU):** if results files already store the docs, no GPU needed.
Run Mode A in your pipeline session; it's the accurate one.

## Step 1 — Load a results file

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json, os, csv

RESULTS_PATH = "/content/drive/MyDrive/crag_results/abl_full.json"   # <-- the run to analyze
CORPUS_RETRIEVER_NAME = "hard_dense"               # retriever used for that run

with open(RESULTS_PATH) as f:
    res = json.load(f)

# auto-detect field names (main-run vs ablation schema)
sample = next(iter(res.values()))
CORRECT_KEY = "crag_correct" if "crag_correct" in sample else "correct"
ANS_KEY     = "crag_ans" if "crag_ans" in sample else "ans"

wrong = {qid: r for qid, r in res.items() if not r[CORRECT_KEY]}
print(f"{len(wrong)} incorrect / {len(res)} total in {os.path.basename(RESULTS_PATH)}")

37 incorrect / 100 total in abl_full.json


## Step 2 — Auto-classify (Mode A: GPU, re-retrieves to locate the answer)
Requires the retriever (e.g. `hard_dense`) and `parse_gold_answers` in the session.
If you only want the CPU approximation, skip to Step 2B.

In [ ]:
retriever = globals().get(CORPUS_RETRIEVER_NAME)

def answer_in(docs, gold):
    blob = " ".join(d.page_content for d in docs).lower()
    return any(g.lower() in blob for g in gold)

def classify(r):
    q, gold, ans = r["question"], r["gold"], r[ANS_KEY]
    path = r.get("path", [])
    refused = ans.strip().lower().startswith("i don't know")
    went_web = any("websearch" in s for s in path)

    if retriever is not None:
        local_docs = retriever.invoke(q)
        in_local = answer_in(local_docs, gold)
    else:
        in_local = None   # can't tell without retriever

    # classification logic
    if in_local and went_web:
        return "grader_misroute"          # had it locally, grader sent it away
    if in_local is False:
        return "retrieval_failure"         # not in local docs
    if refused:
        return "extraction_failure"        # was retrievable but model refused
    return "hallucination"                 # was retrievable, gave wrong confident answer

if retriever is None:
    print("Retriever not found -> use Step 2B (CPU) instead.")
else:
    labels = {qid: classify(r) for qid, r in wrong.items()}
    print("Auto-classification done (Mode A).")

## Step 2B — (CPU fallback) classify by answer-type + path only
Use only if the retriever isn't loaded. Coarser: can't split retrieval vs extraction.

In [3]:
if globals().get(CORPUS_RETRIEVER_NAME) is None:
    def classify_cpu(r):
        ans = r[ANS_KEY].strip().lower()
        went_web = any("websearch" in s for s in r.get("path", []))
        if ans.startswith("i don't know"):
            return "refusal"
        return "wrong+web" if went_web else "wrong+local"
    labels = {qid: classify_cpu(r) for qid, r in wrong.items()}
    print("Auto-classification done (Mode B, CPU approximation).")
else:
    print("Retriever present -> Step 2 (Mode A) already ran; skip this.")

Auto-classification done (Mode B, CPU approximation).


## Step 3 — Distribution table

In [4]:
from collections import Counter
dist = Counter(labels.values())
print(f"{'Category':<22}{'Count':>6}{'%':>7}")
print("-"*36)
for cat, n in dist.most_common():
    print(f"{cat:<22}{n:>6}{n/len(wrong)*100:>6.0f}%")

Category               Count      %
------------------------------------
refusal                   25    68%
wrong+local               10    27%
wrong+web                  2     5%


## Step 4 — Inspect cases per category (for manual review)
Skim these; correct any mislabels by editing `labels[qid]` (e.g. flag noisy_gold).

In [5]:
by_cat = {}
for qid, cat in labels.items():
    by_cat.setdefault(cat, []).append(qid)

for cat, qids in by_cat.items():
    print(f"\n===== {cat}  ({len(qids)} cases) =====")
    for qid in qids[:5]:                  # first 5 per category
        r = wrong[qid]
        print(f"  [{qid}] Q: {r['question']}")
        print(f"        gold: {r['gold']}")
        print(f"        pred: {r[ANS_KEY][:60]!r}")
        print(f"        path: {' -> '.join(r.get('path', []))}")


===== wrong+web  (2 cases) =====
  [5811718] Q: In what city was Stan Andrews born?
        gold: ['Lynn', 'Lynn, Massachusetts']
        pred: 'Chicago, Illinois'
        path: retrieve -> grade=Ambiguous -> websearch -> generate
  [3024339] Q: What genre is Mars?
        gold: ['science fiction film', 'sci-fi film', 'science fiction movie', 'sci-fi movie', 'scifi film', 'scifi movie', 'sci fi film', 'sci fi movie', 'scifi-film', 'scifi']
        pred: 'science fiction'
        path: retrieve -> grade=Incorrect -> websearch -> generate

===== refusal  (25 cases) =====
  [6231364] Q: What genre is You Little Thief?
        gold: ['pop music', 'pop', 'Pop']
        pred: "I don't know."
        path: retrieve -> grade=Incorrect -> websearch -> generate
  [5097922] Q: Who is the author of Mars?
        gold: ['Marc Hempel']
        pred: "I don't know."
        path: retrieve -> grade=Correct -> generate
  [244594] Q: Who was the director of Live and Learn?
        gold: ['Carl Franklin

## Step 5 — Manual corrections (optional)
After reviewing Step 4, fix any mislabels here. Common one: a `retrieval_failure`
that's really a **noisy_gold** case (the gold answer is wrong/ambiguous in PopQA).

In [7]:
# Example manual overrides — edit qids as you review:
# labels["1234567"] = "noisy_gold"
# labels["7654321"] = "noisy_gold"

# (add your corrections above, then re-run Step 3 to see the updated distribution)
print("Apply overrides above, then re-run Step 3 for the corrected distribution.")

Apply overrides above, then re-run Step 3 for the corrected distribution.


## Step 6 — Export labelled CSV for the report

In [9]:
out_path = "/content/error_taxonomy.csv"
with open(out_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "category", "question", "gold", "prediction", "path"])
    for qid, r in wrong.items():
        w.writerow([qid, labels[qid], r["question"], "|".join(r["gold"]),
                    r[ANS_KEY], " -> ".join(r.get("path", []))])
print(f"Exported {len(wrong)} labelled failure cases -> {out_path}")
print("Download from the Colab Files panel for your report appendix.")

Exported 37 labelled failure cases -> /content/error_taxonomy.csv
Download from the Colab Files panel for your report appendix.


## Step 7 — Taxonomy summary (for the writeup)
After review, your error analysis reports:
1. **Distribution** of the N incorrect cases across categories (Step 3 table).
2. **Representative example** per category (from Step 4).
3. **Interpretation**, e.g.:
   - *retrieval_failure* dominant -> the corpus/retriever is the bottleneck.
   - *grader_misroute* present -> evaluator over-rejection (ties to the calibration finding).
   - *hallucination* low -> the "I don't know" guard is working (model rarely fabricates).
   - *noisy_gold* -> caps achievable accuracy; a dataset limitation, not a model failure.